# Step 5: Tuning SVD++ hyperparameters

Your current SVD++ model scored RMSE 0.8617 on hidden data, using
default settings (`n_factors=50, n_epochs=20`). Here we try a small
set of alternative configurations to see if there's easy headroom
left in this model before we add any new complexity.

**The settings we'll vary:**
- `n_factors` — size of each user/movie's learned taste vector.
  More factors can capture richer patterns, but risks overfitting
  on a small dataset and definitely costs more training time.
- `n_epochs` — how many passes over the data during training.
  More epochs = better fit, up to a point of diminishing (or
  negative) returns.
- `reg_all` — regularization strength. Higher values discourage the
  model from fitting noise, which often helps on smaller datasets.

We're doing a small manual grid here (a handful of configs) rather
than an exhaustive search — it's faster to run and easier to read,
and a handful of well-chosen combinations tells us most of what we
need to know.

**One other change in this notebook:** predicting is now done with a
list comprehension instead of `.apply()`. They do the same thing —
call `model.predict()` once per row — but `.apply()` carries extra
overhead per call. This should cut prediction time substantially
versus what you saw before (48.9s -> a few seconds).

In [2]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from surprise import SVDpp, Dataset, Reader

DATA_DIR = "data"
N_USERS_TO_KEEP = 15_000

train_full = pd.read_csv(f"{DATA_DIR}/train.csv")
print(f"Full data: {train_full.shape}")

Full data: (10000038, 4)


In [ ]:
rng = np.random.default_rng(42)
all_users = train_full["userId"].unique()
n_to_keep = min(N_USERS_TO_KEEP, len(all_users)) # Ensure we don't try to keep more users than exist in the dataset
chosen_users = rng.choice(all_users, size=n_to_keep, replace=False) # Select a random subset of users to keep

train = train_full[train_full["userId"].isin(chosen_users)].reset_index(drop=True)
visible, hidden = train_test_split(train, test_size=0.2, random_state=42) # Split the data into visible and hidden sets for training and evaluation

print(f"Visible: {len(visible):,}   Hidden: {len(hidden):,}")

Visible: 744,161   Hidden: 186,041


In [ ]:
reader = Reader(rating_scale=(0.5, 5.0)) # Define the rating scale for the dataset
data = Dataset.load_from_df(visible[["userId", "movieId", "rating"]], reader) # Load the visible data into a Surprise Dataset object for model training
trainset = data.build_full_trainset() # Build the full training set for the model

## Faster prediction helper

Same idea as `model.predict(u, m).est` for every row — just collected
with a list comprehension instead of `.apply()`, which is noticeably
faster in practice for this kind of row-by-row model call.

In [ ]:
# Fast prediction function for Surprise models
# this function takes a trained Surprise model and a DataFrame containing userId and movieId columns, and returns the predicted ratings clipped to the range [0.5, 5.0].
# it uses a list comprehension to iterate over the userId and movieId pairs in the DataFrame, calling the model's predict method for each pair and extracting the estimated rating (est). 
# The resulting predictions are then clipped to ensure they fall within the valid rating range.
def predict_fast(model, df):
    preds = np.array([
        model.predict(u, m).est
        for u, m in zip(df["userId"], df["movieId"])
    ])
    return np.clip(preds, 0.5, 5.0)

## Try a handful of configurations

Each one trains a fresh SVD++ model on the visible set, predicts the
hidden set, and records RMSE plus timing. This will take a few
minutes total since we're training 4 models in sequence.

In [6]:
# Configuration options for the SVD++ model
# Each dictionary in the list represents a different set of hyperparameters to test, including the number of latent factors (n_factors), the number of training epochs (n_epochs), the learning rate (lr_all), and the regularization term (reg_all).
# The results of each configuration will be stored in the results list, which will include the RMSE, training time, and prediction time for each configuration.
# in this example, we are testing four different configurations: a baseline configuration, one with more factors, one with more epochs, and one with stronger regularization.
configs = [
    {"n_factors": 50,  "n_epochs": 20, "lr_all": 0.005, "reg_all": 0.02},  # baseline (what we already ran)
    {"n_factors": 100, "n_epochs": 20, "lr_all": 0.005, "reg_all": 0.02},  # more factors
    {"n_factors": 50,  "n_epochs": 40, "lr_all": 0.005, "reg_all": 0.02},  # more epochs
    {"n_factors": 50,  "n_epochs": 20, "lr_all": 0.005, "reg_all": 0.05},  # stronger regularization
]

results = []
for cfg in configs:
    t0 = time.time()
    model = SVDpp(**cfg, random_state=42)
    model.fit(trainset)
    train_time = time.time() - t0

    t0 = time.time()
    preds = predict_fast(model, hidden)
    pred_time = time.time() - t0

    rmse = np.sqrt(mean_squared_error(hidden["rating"], preds))
    results.append({**cfg, "rmse": rmse, "train_s": train_time, "pred_s": pred_time})

    print(f"{cfg}")
    print(f"  -> RMSE={rmse:.4f}   (train {train_time:.1f}s, predict {pred_time:.1f}s)\n")

{'n_factors': 50, 'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.02}
  -> RMSE=0.8569   (train 878.4s, predict 58.8s)

{'n_factors': 100, 'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.02}
  -> RMSE=0.8615   (train 1265.7s, predict 50.5s)

{'n_factors': 50, 'n_epochs': 40, 'lr_all': 0.005, 'reg_all': 0.02}
  -> RMSE=0.8774   (train 1343.4s, predict 54.7s)

{'n_factors': 50, 'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.05}
  -> RMSE=0.8602   (train 756.9s, predict 50.9s)



## Which configuration won?

In [7]:
# Convert the results to a DataFrame and sort by RMSE
results_df = pd.DataFrame(results).sort_values("rmse")
results_df

,n_factors,n_epochs,lr_all,reg_all,rmse,train_s,pred_s
0,50,20,0.005,0.02,0.856946,878.428352,58.768452
3,50,20,0.005,0.05,0.860189,756.920097,50.906860
1,100,20,0.005,0.02,0.861486,1265.745206,50.492392
2,50,40,0.005,0.02,0.877387,1343.386438,54.667312


# Why lower RMSE wins
RMSE measures how wrong your predictions are, on average, with bigger mistakes penalized more heavily than small ones. It's an error metric, not a score metric... 
A prediction that's off by 0 stars is perfect; a prediction off by 2 stars is bad. RMSE = 0 would mean every single prediction was exactly correct. So we always want this number as small as possible, never as large as possible.
That's why results_df.iloc[0] (the first row after sorting ascending by rmse) is the winner — sort_values("rmse") defaults to ascending order, smallest first.

In [8]:
best = results_df.iloc[0]
print(f"Best config: n_factors={int(best['n_factors'])}, n_epochs={int(best['n_epochs'])}, "
      f"lr_all={best['lr_all']}, reg_all={best['reg_all']}")
print(f"Best RMSE: {best['rmse']:.4f}")
print(f"\nFor comparison:")
print(f"  RMSE — user + movie bias       : 0.8998")
print(f"  RMSE — SVD++ default settings  : 0.8617")
print(f"  RMSE — SVD++ best tuned config : {best['rmse']:.4f}")

Best config: n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02
Best RMSE: 0.8569

For comparison:
  RMSE — user + movie bias       : 0.8998
  RMSE — SVD++ default settings  : 0.8617
  RMSE — SVD++ best tuned config : 0.8569


## What this tells us

If tuning only moves RMSE by a small amount (a few thousandths to
hundredths), that's useful information: SVD++ has mostly hit its
ceiling on this amount of data with this set of features, and the
bigger remaining gains are more likely to come from **more data** or
**more information** (content-based features) than from further
tweaking these specific knobs.

If tuning moves RMSE substantially, it's worth running a wider grid
around whichever direction helped — e.g., if more factors helped,
try 150 or 200 next.

## Next step

With a tuned SVD++ in hand, the next move is bringing in content-based
signal — genome scores and IMDB metadata — and either blending a
second model on top, or testing whether SVD++ alone is already hard
to beat for this amount of data.